#### Create a neural network with one layer and one neuron. The data will be the truth table of AND Gate i.e it will have two inputs and one output. No need to split train/dev/test here. Using the truth table of AND gate trains the neural network. Think about the loss function to be used. Calculate the loss and accuracy. The accuracy of the model must be 100% since the data is very simple. Once the model is trained, save the model and again load the model and use it for prediction.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# 1. Define AND gate data
X = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
y = torch.tensor([[0], [0], [0], [1]], dtype=torch.float32)

In [ ]:
# 2. Define model: 1 layer, 1 neuron
# You define a class ANDGateNN that inherits from nn.Module, which is the base class for all neural networks in PyTorch.
class ANDGateNN(nn.Module):
    def __init__(self):
        super(ANDGateNN, self).__init__() # Call the parent class constructor
        self.fc = nn.Linear(2, 1)  # fully connected linear layer with input_dim=2, output_dim=1

    def forward(self, x):
        # Forward pass of the model. Passes x through the layer and applies a sigmoid to squash the output between 0 and 1 (for binary classification).
        return torch.sigmoid(self.fc(x))  # sigmoid activation

### ✅ PyTorch’s `nn.Module` class handles:

- 📐 **Internal architecture tracking**
- ⚖️ **Weight and parameter management**
- 💾 **Saving/loading models**
- ⚙️ **Many behind-the-scenes utilities**


In [4]:
model = ANDGateNN()

In [ ]:
# 3. Define loss and optimizer

# BCELoss (Binary Cross Entropy) is used for binary classification problems (like AND gate). 
# It measures the difference between predicted values and true binary labels.
criterion = nn.BCELoss()  # Binary Cross-Entropy
optimizer = optim.SGD(model.parameters(), lr=0.1) # model.parameters() gets the weights and biases to update.

In [ ]:
# 4. Train the model
for epoch in range(1000):
    optimizer.zero_grad() # Zero the gradients before the backward pass
    outputs = model(X) # Forward pass: compute the model output
    loss = criterion(outputs, y) # Compute the loss between the model output and the true labels
    loss.backward() # Backward pass: compute the gradients
    optimizer.step() # Update the model parameters

In [ ]:
# 5. Evaluate model
with torch.no_grad(): # Disable gradient calculation for evaluation
    predictions = model(X) # Forward pass to get predictions
    predicted_classes = (predictions > 0.5).float() # Convert probabilities to binary classes (0 or 1)
    accuracy = (predicted_classes == y).float().mean() # Calculate accuracy as the mean of correct predictions
    print(f"Loss: {loss.item():.4f}, Accuracy: {accuracy.item()*100:.2f}%")

Loss: 0.1472, Accuracy: 100.00%


In [ ]:
# 6. Save the model
torch.save(model.state_dict(), 'and_gate_model.pth') # Save the model's state dictionary (weights and biases) to a file

In [ ]:
# 7. Load the model
loaded_model = ANDGateNN() # Create a new instance of the model
loaded_model.load_state_dict(torch.load('and_gate_model.pth')) # Load the saved state dictionary into the new model instance
loaded_model.eval() # Set the model to evaluation mode (disables dropout and batch normalization if used)

# 8. Use loaded model for prediction
with torch.no_grad(): # Disable gradient calculation for inference
    pred = loaded_model(X) # Forward pass to get predictions
    
    print("Predictions:", torch.round(pred).view(-1).tolist()) # Print the predictions as a list of 0s and 1s
    # view(-1) flattens the tensor for printing.

Predictions: [0.0, 0.0, 0.0, 1.0]


C:\Users\user\AppData\Local\Temp\ipykernel_20456\2143976369.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load('and_gate_model.pth')

## 🔍 Why use Binary Cross Entropy Loss for AND Gate?

Because:

- The **AND gate** is a **binary classification problem** (output: 0 or 1).
- The model outputs **probabilities using the sigmoid activation function**.
- We want a loss function that:
  - ✅ **Penalizes incorrect probabilities**.
  - ✅ **Encourages the predicted probability to be close to the true label (0 or 1)**.
